<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Deep RL: el Cerebro, las Fichas de Estudio y el Profesor Paciente 🤖
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/06%20-%20Aprendizaje%20por%20Refuerzo/Para%20Dummies/03_Deep_RL_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 03 (Deep RL: DQN y gradientes de política con redes)](../03_Deep_RL_DQN_y_Policy_Gradient.ipynb). La libreta de notas funciona en un mundo pequeño, pero **no en un videojuego**, donde las situaciones posibles son más que los átomos del universo. La solución: reemplazar la libreta por un **cerebro artificial** (una red neuronal) que **reconoce situaciones parecidas**. Es lo que hizo famoso a *DQN*, el sistema que aprendió a jugar videojuegos de Atari **mirando solo la pantalla**. Aquí **no necesitas** instalar nada especial: usamos solo *NumPy* y gráficas.

Al terminar podrás explicar:
1. Por qué **la libreta ya no cabe** y qué aporta un «cerebro» además de ahorrar espacio.
2. Qué son las **fichas de estudio barajadas** (*memoria de repetición*) y por qué evitan que el agente **olvide**.
3. Qué es el **profesor paciente** (*red objetivo*) y qué problema resuelve.
4. Por qué entrenar con refuerzo profundo es **inestable**: dos intentos con la misma receta no dan el mismo resultado.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../03_Deep_RL_DQN_y_Policy_Gradient.ipynb). Antes: [La perilla de las probabilidades (Dummies)](02_Gradientes_de_Politica_Dummies.ipynb).

---
## 1. Cuando la libreta no cabe 📚💥

Una libreta con **una nota por situación** es perfecta para un lago de 16 casillas. Pero mira cómo crece el número de «situaciones» cuando el mundo se parece más a la realidad:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Numero de "situaciones" distintas que tendria que anotar la libreta (en potencias de 10)
casos = {
    "Lago congelado 4x4": np.log10(16),
    "Tres-en-raya (aprox.)": np.log10(5478),
    "Carrito con palo: 4 medidas, 100 valores c/u": 4 * 2.0,
    "Una foto de Atari (210x160 píxeles, color)": 210 * 160 * 3 * np.log10(256),
}
for nombre, e in casos.items():
    print(f"{nombre:<45} ~ 10^{e:,.0f} situaciones" if e > 8 else f"{nombre:<45} ~ {10 ** e:,.0f} situaciones")

fig, ax = plt.subplots(figsize=(10, 3.8))
nombres = list(casos); valores = [casos[n] for n in nombres]
ax.barh(nombres[::-1], valores[::-1], color=["#dc2626", "#f59e0b", "#2563eb", "#16a34a"])
ax.set_xscale("symlog"); ax.axvline(80, color="black", ls="--"); ax.text(85, 0.1, "átomos del universo\n(~10^80)", fontsize=8, va="bottom")
ax.set_xlabel("Exponente: situaciones = 10^(este número)  (escala comprimida)")
ax.set_title("La libreta explota: ni todo el universo alcanzaría para anotarlo", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert casos["Una foto de Atari (210x160 píxeles, color)"] > 242000

---
Y no es solo cuestión de espacio. Una libreta trata cada situación como **independiente**: si aprendiste algo en una esquina, **no le sirve a la esquina de al lado**, aunque sean casi idénticas. Un **cerebro artificial** (una *red neuronal*) hace algo parecido a lo que hacemos las personas: **reconoce parecidos**. Si dos pantallas del videojuego casi no cambian, el cerebro les da **notas parecidas**, sin que tenga que haber visto exactamente esa pantalla antes. Eso se llama **generalizar**.

El precio: **lo que antes era seguro (la libreta) ahora es frágil**. El cerebro se ajusta con ejemplos y **puede estropear lo que ya sabía** al aprender algo nuevo. Los creadores de DQN idearon dos «trucos» para domesticar eso. El primero:

## 2. Las fichas de estudio barajadas 🃏

Imagina que estudias para un examen de **toda la materia**, pero tu cuaderno te obliga a estudiar **en orden, un tema por semana**, y nunca volver atrás. Cuando llegas al último tema, **ya olvidaste el primero**. Lo sensato es hacer **fichas de estudio** de todo lo que has visto y **barajarlas**, para repasar **un poco de todo** en cada sesión.

Un agente que juega vive algo parecido: lo que ve **en un momento** es muy parecido a lo que vio **hace un segundo**. Si el cerebro aprende solo de lo más reciente, **olvida lo anterior**. DQN guarda sus experiencias en una **memoria** («fichas») y entrena con **grupos al azar** de esa memoria (*memoria de repetición*). Lo comprobamos con un **experimento pequeño**: el «cerebro» debe aprender una curva ondulada, y la experiencia le llega **de izquierda a derecha, en orden**. Comparamos aprender **solo con lo último que vio** frente a **barajar las fichas guardadas**.

In [ ]:
rng = np.random.default_rng(0)
curva = lambda x: np.sin(2 * np.pi * x)                               # lo que el "cerebro" debe aprender
centros = np.linspace(0, 1, 15)
rasgos = lambda x: np.exp(-((x[:, None] - centros[None, :]) ** 2) / (2 * 0.06 ** 2))      # un "cerebro" simple: suma de campanas

N = 600
x = np.linspace(0, 1, N); y = curva(x) + rng.normal(0, 0.05, N)       # la experiencia llega EN ORDEN (de izquierda a derecha)
x_prueba = np.linspace(0, 1, 400)

def aprender(modo, tamano_grupo=20, paso=0.5, semilla=0):
    r = np.random.default_rng(semilla); pesos = np.zeros(len(centros)); errores = []
    for t in range(N):
        if modo == "solo lo ultimo":   idx = np.arange(max(0, t - tamano_grupo + 1), t + 1)          # estudia solo lo mas reciente
        else:                          idx = r.integers(0, t + 1, tamano_grupo)                       # baraja TODAS las fichas guardadas
        F = rasgos(x[idx]); pesos -= paso * F.T @ (F @ pesos - y[idx]) / len(idx)
        if t % 50 == 49: errores.append(np.sqrt(np.mean((rasgos(x_prueba) @ pesos - curva(x_prueba)) ** 2)))
    return pesos, np.array(errores)

w_ult, e_ult = aprender("solo lo ultimo"); w_bar, e_bar = aprender("barajar fichas")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
axes[0].plot(x_prueba, curva(x_prueba), color="black", lw=2, label="La curva real"); axes[0].scatter(x[::6], y[::6], s=8, color="gray", alpha=0.5, label="Experiencia (llega en orden)")
axes[0].plot(x_prueba, rasgos(x_prueba) @ w_ult, color="#dc2626", lw=2.5, label="Solo con lo último que vio")
axes[0].plot(x_prueba, rasgos(x_prueba) @ w_bar, color="#16a34a", lw=2.5, label="Barajando las fichas guardadas")
axes[0].set_title("Lo que aprendió cada «cerebro» al final", fontweight="bold", fontsize=10); axes[0].legend(fontsize=8)
axes[1].plot(np.arange(1, len(e_ult) + 1) * 50, e_ult, color="#dc2626", lw=2.5, label="Solo lo último"); axes[1].plot(np.arange(1, len(e_bar) + 1) * 50, e_bar, color="#16a34a", lw=2.5, label="Fichas barajadas")
axes[1].set_xlabel("Experiencias recibidas"); axes[1].set_ylabel("Error al predecir TODA la curva"); axes[1].legend()
axes[1].set_title("Con las fichas barajadas el error baja casi a cero", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Error final: solo con lo último = {e_ult[-1]:.3f} | con fichas barajadas = {e_bar[-1]:.3f}")
assert e_bar[-1] < 0.1 and e_ult[-1] > 0.2                               # olvidar el principio cuesta caro; barajar lo evita

---
## 3. El profesor paciente 🧑‍🏫

El segundo truco. Imagina que un alumno corrige sus respuestas comparándolas con las de **un profesor... que cambia sus respuestas cada vez que el alumno cambia las suyas** (¡porque el profesor **es** el propio alumno!). El alumno persigue una meta que **se mueve cada vez que se acerca**: es como **perseguir tu propia sombra**. El aprendizaje se vuelve inestable.

La solución de DQN: tener **dos copias** del cerebro.

| Pieza de DQN | La analogía | Qué problema resuelve |
|---|---|---|
| **Red** (cerebro) | El alumno | Reconocer situaciones parecidas (generalizar) |
| **Memoria de repetición** | **Fichas de estudio barajadas** | El agente **olvida** lo viejo si solo repasa lo último |
| **Red objetivo** | **Un profesor paciente**: una copia del alumno que solo se **actualiza de vez en cuando** (cada cierto número de pasos) | La meta **no se mueve** en cada paso: el alumno persigue un blanco **quieto** un rato |
| **Probar al azar al principio y menos después** | Explorar mucho cuando no sabes nada | El dilema de probar vs. repetir |
| **No exagerar con errores enormes** (pérdida de Huber) | No desmoralizarse por un solo error gigante | Un solo error enorme no estropea todo lo aprendido |

Con estas piezas DQN llegó a jugar muchos juegos de Atari a un nivel comparable al de una persona (*Mnih et al., 2015*; el capítulo de Géron cuenta que superó a las personas en la mayoría de esos juegos). Pero **no hay garantías**: son «parches» que funcionan **la mayoría de las veces**.

## 4. Cada intento sale un poco distinto 🎰

Una verdad incómoda del refuerzo: **entrenas dos veces con la misma receta y no obtienes exactamente lo mismo** (por suerte inicial, por qué situaciones se probaron primero...). Lo vemos con **datos reales**: entrenamos **8 veces** al mismo agente del lago congelado (con su libreta de notas, más simple que un cerebro artificial) y miramos qué porcentaje de partidas gana en cada etapa del entrenamiento. **Ojo con la lectura honesta:** aquí los 8 intentos se parecen bastante, porque el lago es pequeño y la libreta es estable; con **cerebros artificiales** la dispersión suele ser **mucho mayor** (lo verás con 3 semillas de DQN en el cuaderno estándar, donde, en la ejecución con la que se preparó este cuaderno, las evaluaciones de la política voraz variaban entre ~60 y ~290 puntos según el momento y la semilla).

In [ ]:
MAPA = ["SFFF", "FHFH", "FFFH", "HFFG"]; MOV = [(0, -1), (1, 0), (0, 1), (-1, 0)]

import random

def entrenar_lago(semilla, episodios=6000, pasito=0.1):
    """Un agente con libreta (Q-learning) en el lago resbaloso. Devuelve, por partida, 1 si llego al tesoro y 0 si no."""
    azar = random.Random(semilla); notas = [[0.0] * 4 for _ in range(16)]; aciertos = np.zeros(episodios)
    for ep in range(episodios):
        eps = max(0.05, 1.0 - ep / 3000)                          # al principio prueba mucho; despues cada vez menos
        f = c = 0
        for _ in range(100):
            q = notas[4 * f + c]
            if azar.random() < eps: a = azar.randrange(4)
            else: a = azar.choice([i for i in range(4) if q[i] == max(q)])
            d = (a + azar.choice((-1, 0, 1))) % 4                   # piso resbaloso: 1/3 lo que querias, 1/3 a cada lado
            f2 = min(max(f + MOV[d][0], 0), 3); c2 = min(max(c + MOV[d][1], 0), 3)
            fin = MAPA[f2][c2] in "HG"; r = 1.0 if MAPA[f2][c2] == "G" else 0.0
            futuro = 0.0 if fin else max(notas[4 * f2 + c2])
            q[a] += pasito * (r + 0.99 * futuro - q[a])
            f, c = f2, c2
            if fin:
                aciertos[ep] = r; break
    return aciertos

N_INTENTOS, VENT = 8, 250
curvas = np.array([np.convolve(entrenar_lago(k), np.ones(VENT) / VENT, mode="valid") for k in range(N_INTENTOS)]) * 100
fig, ax = plt.subplots(figsize=(10, 4.6))
for k in range(N_INTENTOS):
    ax.plot(np.arange(curvas.shape[1]) + VENT, curvas[k], lw=1.3, alpha=0.7, label=f"intento {k + 1}")
ax.plot(np.arange(curvas.shape[1]) + VENT, curvas.mean(axis=0), color="black", lw=3, label="promedio")
ax.axhline(1.5, color="gray", ls=":", label="al azar (~1.5%)")
ax.set_xlabel("Partidas jugadas"); ax.set_ylabel("% de partidas que llegan al tesoro (últimas 250)")
ax.set_title("La misma receta, 8 intentos: cada uno aprende a su ritmo", fontweight="bold"); ax.legend(fontsize=7, ncol=3)
plt.tight_layout(); plt.show()

finales = curvas[:, -1]
print(f"Al final del entrenamiento (partidas ~5750-6000): mejor intento = {finales.max():.0f}%, peor intento = {finales.min():.0f}%, promedio = {finales.mean():.0f}%")
assert finales.mean() > 15 and finales.min() > 10 and finales.max() - finales.min() > 5          # todos mejoran respecto al azar, pero NO son identicos
print("Lección: ningun intento es identico a otro (aqui las diferencias son moderadas); en refuerzo profundo son mucho mayores, asi que nunca juzgues una receta por UN solo intento: mira varios y reporta tambien los malos.")

---
##### 🛠️ Práctica 1: Arma tu propia memoria de fichas

Construyamos a mano la **memoria de fichas** de DQN, que solo guarda las experiencias más recientes. Imagina que cada ficha es un número (la experiencia 1, la 2, ...).

1. Crea `CAPACIDAD = 5` y una lista vacía `memoria = []`.
2. Recibe las experiencias numeradas del **1 al 8**, una por una: agrégala al final de la lista y, si la lista ya tiene más de 5 fichas, **tira la más vieja** (`memoria.pop(0)`).
3. Comprueba que al final `memoria == [4, 5, 6, 7, 8]` (solo quedan las 5 más recientes).
4. **Baraja:** con `rng_fichas = np.random.default_rng(0)` elige **3 posiciones distintas** de la memoria (`rng_fichas.choice(len(memoria), size=3, replace=False)`) y guarda esas fichas en la lista `grupo`. Comprueba que son 3 fichas diferentes y que todas están en la memoria.

In [ ]:
# Escribe tu código aquí

CAPACIDAD = 5
memoria = []

# 2) Recibir las experiencias 1, 2, ..., 8
# for experiencia in range(1, 9):
#     memoria.append(experiencia)
#     if len(memoria) > ...:
#         memoria.pop(0)                 # tirar la ficha mas vieja

# 4) Barajar: elegir 3 fichas al azar, sin repetir
# rng_fichas = np.random.default_rng(0)
# posiciones = rng_fichas.choice(len(memoria), size=3, replace=False)
# grupo = [memoria[i] for i in posiciones]

# 3) y 4) Comprobaciones con assert


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
CAPACIDAD = 5
memoria = []

# 2) Recibir las experiencias 1, 2, ..., 8
for experiencia in range(1, 9):
    memoria.append(experiencia)
    if len(memoria) > CAPACIDAD:
        memoria.pop(0)                 # tirar la ficha mas vieja
print("Memoria al final:", memoria)

# 3) Solo quedan las 5 mas recientes
assert memoria == [4, 5, 6, 7, 8]

# 4) Barajar: elegir 3 fichas al azar, sin repetir
rng_fichas = np.random.default_rng(0)
posiciones = rng_fichas.choice(len(memoria), size=3, replace=False)
grupo = [memoria[i] for i in posiciones]
print("Grupo de estudio de hoy:", grupo)
assert len(set(grupo)) == 3 and all(f in memoria for f in grupo)
print("La memoria olvida lo mas viejo y cada sesion de estudio mezcla fichas de distintos momentos.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Cuando el mundo es demasiado grande para una libreta, se usa un *cerebro artificial* que generaliza; para que no se desestabilice se le dan *fichas de estudio barajadas* (memoria de repetición) y un *profesor paciente* (red objetivo), y aun así el entrenamiento es frágil: cada intento sale distinto, así que hay que probar varias veces antes de creer en un resultado.**

### Lo que aprendiste hoy:
- ✅ La libreta **explota** en mundos reales; un cerebro artificial **generaliza** a situaciones parecidas.
- ✅ **Fichas barajadas**: repasar un poco de todo evita **olvidar** lo viejo (el experimento lo mostró: error ~0 frente a ~0.3).
- ✅ **Profesor paciente**: una copia que cambia despacio da una meta **quieta** para perseguir.
- ✅ **No hay garantías**: estas técnicas son «parches» que suelen funcionar.
- ✅ **Cada intento sale un poco distinto** (y con cerebros artificiales mucho más): reporta varias semillas, incluidos los malos intentos.

> 🎓 **Cierre del módulo:** ahora puedes explorar el [cuaderno estándar](../03_Deep_RL_DQN_y_Policy_Gradient.ipynb), donde se entrena un DQN real en *CartPole*, y continuar con el [Taller Práctico](../../homeworks/06_Aprendizaje_por_Refuerzo_Hands_On.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿para qué sirven las «fichas de estudio barajadas» y qué pasaría si el agente aprendiera solo de lo último que vivió? Pon un ejemplo de tu vida (estudiar, aprender un instrumento).
2. Un amigo entrena su robot **una vez**, le va muy bien y dice «mi método es el mejor». ¿Qué le responderías y qué le pedirías que haga antes de creerle?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>